# ADK による AI エージェント開発の基礎

このノートブックでは、ADK で会話型の AI エージェントを作成・利用する基本的な手順を確認します。

## 事前準備

**[ADB-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[ADB-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[ADB-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[ADB-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [2]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[ADB-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import HTML, Markdown, display
import vertexai
from vertexai.agent_engines import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools import google_search

vertexai.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[ADB-06]**

Grounding with Google Search を利用して、ユーザーの質問に回答する AI エージェント（LlmAgent オブジェクト）を作成します。

In [5]:
instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- google_search を使用して、最新情報に基づいて回答してください。
- フレンドリーな会話を心がけてください。
'''

search_agent = LlmAgent(
    name='search_agent',
    model='gemini-3.5-flash-lite',
    description='Google検索を用いて質問に回答するエージェント',
    instruction=instruction,
    tools=[google_search],
)


**[ADB-07]**

作成した LlmAgent オブジェクトを含む AdkApp オブジェクトを作成します。

このオブジェクトのメソッドを通じて、AI エージェントと対話します。

In [6]:
search_agent_app = AdkApp(
    agent=search_agent,
    app_name='search_agent_app',
)

## AdkApp オブジェクトと会話するアプリケーションの作成

**[ADB-08]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [7]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = getattr(session, 'id', None) or session['id']

        result = []
        events = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            events.append(event)
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result), events

## 会話に伴うイベントデータの確認

**[ADB-09]**

ChatClient クラスのインスタンスを作成して、ユーザーのメッセージを送信します。

変数 `response` に AI エージェントの応答メッセージがマークダウンテキストで格納されます。

In [15]:
chat_client = ChatClient(search_agent_app)

query = '''
高田馬場のおすすめのカレー屋は？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

高田馬場は、早稲田大学がある学生街ということもあり、**ハイレベルで個性的なカレー激戦区**として知られています！その中でも特におすすめの人気店をいくつか厳選してご紹介しますね。

---

### 1. カレーライス専門店 ブラザー
* **特徴**: 高田馬場を代表する大行列のスパイスカレーの名店。
* **おすすめポイント**: 定番のチキンカレーや欧風カレーに加え、他ではなかなか味わえない**「鯖キーマ（サバキーマ）」**が圧倒的な人気を誇ります。スパイスの旨味と鯖の風味が絶妙にマッチしており、一度食べたら癖になる美味しさです。

### 2. プネウマカレー
* **特徴**: 驚異的なコスパとこだわりが光る、知る人ぞ知る人気店。
* **おすすめポイント**: 伊吹島産のアンチョビを隠し味に使った、スパイシーでコクのあるチキンカレーが看板メニューです。毎朝挽きたてのスパイスを使用していながら、お手頃価格でボリュームも満点。サクッと美味しいカレーを食べたい時にぴったりです（売り切れ次第終了なので早めの訪問がおすすめ）。

### 3. カリーライス専門店 エチオピア 高田馬場店
* **特徴**: 神保町に本店を構える、薬膳スパイスたっぷりの名店の支店。
* **おすすめポイント**: サラッとしたインド風のルーに、何十種類ものスパイスが溶け込んだ深みのある味わいが魅力です。特に、ホクホクの豆がたっぷり入った「豆カレー」がヘルシーかつ絶品でファンが多いです。辛さも細かく選べるので、辛口好きにもおすすめです。

### 4. Biryani Tokyo（ビリヤニ トウキョウ）
* **特徴**: 本格的なビリヤニ（スパイス炊き込みご飯）を堪能できる専門店。
* **おすすめポイント**: カレーライスとはひと味違う、スパイスと肉・米のハーモニーを楽しみたい時におすすめです。香り高い本格的なビリヤニが高田馬場でも気軽に味わえると評判です。

---

王道のスパイスカレーなら「ブラザー」、サクッと安くて美味しいカレーなら「プネウマカレー」、じっくりスパイスを感じたいなら「エチオピア」など、その日の気分に合わせて選んでみてくださいね！気になるお店はありましたか？

**[ADB-10]**

変数 `events` には、AI エージェントの処理過程の情報を含むさまざまなイベントデータが格納されています。

>ファンクションコールなど、多段階のループ処理を行った場合は、複数のイベントがリスト形式で格納されますが、今回の場合は、単一のイベント `evetns[0]` のみが含まれます。

ディクショナリ形式のイベントデータに含まれる Key を確認します。

In [16]:
events[0].keys()

dict_keys(['model_version', 'content', 'grounding_metadata', 'finish_reason', 'usage_metadata', 'invocation_id', 'author', 'actions', 'node_info', 'id', 'timestamp'])

**[ADB-11]**

特に `author` と `timestamp` には、イベントを発行した LlmAgent オブジェクトの名前と、タイムスタンプが格納されています。

In [17]:
author = events[0]['author']
timestamp = events[0]['timestamp']
print(f'{author}: {datetime.fromtimestamp(timestamp, tz=ZoneInfo('Asia/Tokyo'))}')

search_agent: 2026-08-31 14:09:59.030866+09:00


**[ADB-12]**

`content` には、応答メッセージが含まれます。

In [18]:
events[0]['content']

{'parts': [{'text': '高田馬場は、早稲田大学がある学生街ということもあり、**ハイレベルで個性的なカレー激戦区**として知られています！その中でも特におすすめの人気店をいくつか厳選してご紹介しますね。\n\n---\n\n### 1. カレーライス専門店 ブラザー\n* **特徴**: 高田馬場を代表する大行列のスパイスカレーの名店。\n* **おすすめポイント**: 定番のチキンカレーや欧風カレーに加え、他ではなかなか味わえない**「鯖キーマ（サバキーマ）」**が圧倒的な人気を誇ります。スパイスの旨味と鯖の風味が絶妙にマッチしており、一度食べたら癖になる美味しさです。\n\n### 2. プネウマカレー\n* **特徴**: 驚異的なコスパとこだわりが光る、知る人ぞ知る人気店。\n* **おすすめポイント**: 伊吹島産のアンチョビを隠し味に使った、スパイシーでコクのあるチキンカレーが看板メニューです。毎朝挽きたてのスパイスを使用していながら、お手頃価格でボリュームも満点。サクッと美味しいカレーを食べたい時にぴったりです（売り切れ次第終了なので早めの訪問がおすすめ）。\n\n### 3. カリーライス専門店 エチオピア 高田馬場店\n* **特徴**: 神保町に本店を構える、薬膳スパイスたっぷりの名店の支店。\n* **おすすめポイント**: サラッとしたインド風のルーに、何十種類ものスパイスが溶け込んだ深みのある味わいが魅力です。特に、ホクホクの豆がたっぷり入った「豆カレー」がヘルシーかつ絶品でファンが多いです。辛さも細かく選べるので、辛口好きにもおすすめです。\n\n### 4. Biryani Tokyo（ビリヤニ トウキョウ）\n* **特徴**: 本格的なビリヤニ（スパイス炊き込みご飯）を堪能できる専門店。\n* **おすすめポイント**: カレーライスとはひと味違う、スパイスと肉・米のハーモニーを楽しみたい時におすすめです。香り高い本格的なビリヤニが高田馬場でも気軽に味わえると評判です。\n\n---\n\n王道のスパイスカレーなら「ブラザー」、サクッと安くて美味しいカレーなら「プネウマカレー」、じっくりスパイスを感じたいなら「エチオピア」など、その日の気分に合わせて選んでみてくださいね！気になるお店はありましたか？',
   't

**[ADB-13]**

Grounding with Google Search を使用した場合は、Google 検索に使用したキーワードも確認できます。

In [19]:
events[0]['grounding_metadata']['web_search_queries']

['高田馬場 おすすめ カレー', '高田馬場 カレー 人気店']

**[ADB-14]**

同じキーワードで検索を実行するボタンを表示する HTML テキストも用意されます。

In [20]:
display(HTML(
    events[0]['grounding_metadata']['search_entry_point']['rendered_content']
))

**[ADB-15]**

これまでの会話履歴は、AdkApp オブジェクト内の SessionService オブジェクトに保存されているので、そのまま会話を継続できます。

In [21]:
query = '''
特に家族連れにおすすめなのは？
'''
response, events = await chat_client.async_stream_query(query)
display(Markdown(response))

ご家族連れ（小さなお子様からおじいちゃん・おばあちゃんまで）で高田馬場に行く場合、先ほどご紹介したスパイス専門店（「ブラザー」や「プネウマカレー」など）はカウンター席中心で常時行列ができていたり、激辛・本格スパイスが強すぎたりするため、少しハードルが高くなります。

そのため、ご家族連れであれば、以下のような**テーブル席があって辛さの調整ができ、子供も食べやすいメニューがあるお店**がおすすめです！

### 1. タンドール料理やインド・ネパール系のナン＆カレーのお店
高田馬場には本格的なインド・ネパール料理店（「Namaste Asian Dining ＆ Bar」や周辺の多国籍系レストランなど）がたくさんあります。
* **おすすめ理由**: 
  * 広めのテーブル席があるところが多く、ベビーカーの入店や子供連れでも比較的受け入れてもらいやすいです。
  * 何より、**甘口のバターチキンカレー**や、子供が大好きな**焼きたての大きなナン**が選べるので、辛いものが食べられないお子様でも安心して楽しめます。

### 2. デパート・商業施設や、少し落ち着いたロードサイド・ホテル系のお店
高田馬場駅から少し離れますが、例えば少し足を伸ばして「リーガロイヤルホテル東京」（早稲田・高田馬場エリア）のダイニング等であれば、ホテルの上質な空間でゆったりと美味しい洋食やホテルメイドのカレーを家族で楽しむことができます。

---

**💡 家族連れでのお店選びのコツ**
もし高田馬場の街中で探す場合は、狭いカウンター席の専門店を避け、**「テーブル席があるインド料理店（インネパ系）」**や、ファミリー層も入りやすい**チェーンの洋食・カレー店、または駅周辺の少し広めのお店**を狙うと、周りを気にせずゆったり食事ができるので安心ですよ！

**[ADB-16]**

今回の応答メッセージの生成に使用した検索キーワードを確認します。

In [22]:
events[0]['grounding_metadata']['web_search_queries']

['高田馬場 カレー 家族連れ 子供連れ', '高田馬場 カレー 子連れ おすすめ']